In [11]:
# import libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.formula.api as smf
import bambi as bmb
import arviz as az
import polars as pl

In [12]:
# Set randomization seed
rng = np.random.default_rng(42)

In [13]:
# Specify a function to simulate data
# predictors: snow making (in), # of days open last year, # of runs at the resort, avg snow fall(in)
def sim_data(n, beta_0, beta_snow_making, beta_days_open_last, beta_runs, beta_avgsnow, sigma):
    
    snow_making = rng.normal(2, 700, size=n).clip(min=0)
    days_open_last = rng.normal(45, 165, size=n).clip(min=0)
    runs = rng.normal(13, 336, size=n).clip(min=1)
    avgsnow = rng.normal(18, 550, size=n).clip(min=0)
    error = rng.normal(0, sigma, size=n)

    sentiment_latent = ( #creating the sentiment function
        beta_0
        + beta_snow_making * snow_making
        + beta_days_open_last * days_open_last
        + beta_runs * runs
        + beta_avgsnow * avgsnow
        + error
    )
    sentiment = np.clip(np.round(sentiment_latent), 1, 7)  # editing sentiment_latent to fit the model to the 7-point scale
    return sentiment, snow_making, days_open_last, runs, avgsnow, error

sentiment, snow_making, days_open_last, runs, avgsnow, error = sim_data( # run coefficients based on simulation estimates
    n=200, beta_0=-1.0, beta_snow_making=0.002, beta_days_open_last=0.02,
    beta_runs=0.015, beta_avgsnow=0.002, sigma=0.8
)

In [14]:
# create dataframe for storage
df = pd.DataFrame({
   "sentiment": sentiment,
    "snow_making": snow_making,
    "days_open_last": days_open_last,
    "runs": runs,
    "avgsnow": avgsnow,
    "error": error
})

In [15]:
print (df.head())

   sentiment  snow_making  days_open_last        runs     avgsnow     error
0        1.0   215.301956      100.699801    1.000000  301.475722 -0.676036
1        5.0     0.000000      277.234507   79.116768    0.000000 -0.866959
2        7.0   527.315837       59.946510  288.697568  718.945969  0.275684
3        4.0   660.395301      151.249901    1.000000    0.000000  0.303424
4        3.0     0.000000        0.000000  188.112198    0.000000  1.029865


In [16]:
#model fitting 
fr_fit = smf.ols ( 
    "sentiment ~ 1 + snow_making + days_open_last + runs + avgsnow", data = df).fit()

In [17]:
#Display regression results 
print(fr_fit.summary())

                            OLS Regression Results                            
Dep. Variable:              sentiment   R-squared:                       0.799
Model:                            OLS   Adj. R-squared:                  0.795
Method:                 Least Squares   F-statistic:                     193.4
Date:                Wed, 23 Sep 2026   Prob (F-statistic):           1.06e-66
Time:                        18:21:23   Log-Likelihood:                -296.83
No. Observations:                 200   AIC:                             603.7
Df Residuals:                     195   BIC:                             620.1
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          0.7687      0.135      5.